# Spike del modelo de lenguaje · INV-30

Compara los modelos candidatos para el agente de InventAI/o con el conjunto v0: 43 preguntas, 20 base y 23 variantes del usuario. El diseño está en `docs/INV-30-requerimientos.md` §4.4 y en la §5.1 de `docs/SPRINT-6-requerimientos.md`.

**Dónde corre**
- **En el equipo de desarrollo** (VS Code, kernel `~/venvs/inventaio/bin/python`): Ollama nativo en WSL (`bash scripts/ollama_wsl.sh iniciar`) con `qwen3:8b`, `qwen3:4b-instruct-2507` y `llama3.1:8b`. **Es la medición oficial**, porque la demostración del Release 2 corre aquí.
- **En Colab** (Entorno de ejecución → Cambiar tipo → **L4**): instala Ollama 0.40.5 desde su release de GitHub, clona la rama del spike y baja los mismos tres modelos. Además, como techo de calidad, `qwen3:14b` y `qwen3:30b-a3b-instruct-2507`, que no caben en 8 GB.

**Para que Colab y el equipo se puedan comparar**, la primera celda y la de modelos lo verifican:
- la **misma versión de Ollama** (0.40.5);
- los **mismos pesos** de cada modelo, con el digest igual al del equipo;
- el **mismo commit** del arnés, que queda anotado en los resultados;
- la **misma variante** del conjunto. En el equipo, `VARIANTE = "colab"` corre exactamente lo mismo que Colab.

**Regla de los datos (N9, D22).** En Colab corre la variante `colab` del conjunto: sucursales ficticias (NORTE, CENTRO, SUR), otra fecha y cantidades sintéticas. Nunca se usa la bodega real fuera del equipo, y ningún modelo corre en servidores externos (`OLLAMA_NO_CLOUD=1`).

**Partes:** 1) entorno · 2) modelos · 3) explorar preguntas · 4) examen completo, en CSV · 5) rendimiento · 6) descargar los resultados.

## 1. Entorno

In [ ]:
import os, sys, subprocess, time, json, glob, shutil
from datetime import datetime
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
RAMA = "feature/INV-30-nlp-agent"       # develop, después del merge del spike
OLLAMA_VERSION = "0.40.5"               # la del equipo de desarrollo: con otra versión los resultados no se comparan
REPO_GIT = "https://github.com/ddelgadillod/InventaIO.git"
VARIANTE_EN_EL_EQUIPO = "local"         # "colab" para correr en el equipo exactamente lo mismo que en Colab

def sh(comando):
    """Corre un comando y devuelve su salida; si falla, muestra el error completo."""
    r = subprocess.run(comando, shell=isinstance(comando, str), capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Falló: {comando}\n{r.stdout}\n{r.stderr}")
    return r.stdout.strip()

if EN_COLAB:
    # Fuera del repositorio: así se puede volver a clonar sin dejar al kernel en una carpeta borrada
    os.chdir("/content")
    REPO = "/content/InventaIO"
    rama_clonada = sh(["git", "-C", REPO, "rev-parse", "--abbrev-ref", "HEAD"]) if os.path.isdir(REPO + "/.git") else None
    if rama_clonada != RAMA:
        shutil.rmtree(REPO, ignore_errors=True)
        sh(["git", "clone", "--depth", "1", "--branch", RAMA, REPO_GIT, REPO])
    else:
        sh(["git", "-C", REPO, "pull", "--ff-only"])
    sh([sys.executable, "-m", "pip", "install", "-q", "httpx", "pydantic-settings"])

    # Ollama desde su release de GitHub. Desde la 0.5 se publica como .tar.zst (no .tgz) y hace
    # falta zstd, que Colab no trae: sin esto se termina instalando una versión vieja sin Qwen3.
    # Se verifica el SHA-256 y se borra cualquier otra instalación que la tape en el PATH.
    if OLLAMA_VERSION not in subprocess.run("/usr/bin/ollama --version", shell=True, capture_output=True, text=True).stdout:
        sh("apt-get -qq update && apt-get -qq install -y zstd")
        base = f"https://github.com/ollama/ollama/releases/download/v{OLLAMA_VERSION}"
        sh(f"curl -fsSL --retry 5 -o /tmp/ollama-linux-amd64.tar.zst {base}/ollama-linux-amd64.tar.zst")
        sh(f"curl -fsSL --retry 5 -o /tmp/sha256sum.txt {base}/sha256sum.txt")
        sh("cd /tmp && grep 'ollama-linux-amd64.tar.zst$' sha256sum.txt | sed 's| ./| |' | sha256sum -c -")
        sh("rm -rf /usr/lib/ollama && tar --use-compress-program=unzstd -xf /tmp/ollama-linux-amd64.tar.zst -C /usr")
        sh("rm -f /usr/local/bin/ollama /tmp/ollama-linux-amd64.tar.zst")

    subprocess.run("pkill -f 'ollama serve'", shell=True)
    time.sleep(2)
    entorno_ollama = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_NO_CLOUD": "1", "OLLAMA_NUM_PARALLEL": "1",
                      "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_KEEP_ALIVE": "30m"}
    subprocess.Popen(["/usr/bin/ollama", "serve"], env=entorno_ollama,
                     stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
    VARIANTE = "colab"
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    ENTORNO = "colab-" + (gpu.split()[-1].lower() if gpu else "cpu")
else:
    REPO = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    VARIANTE = VARIANTE_EN_EL_EQUIPO
    ENTORNO = "local" if VARIANTE == "local" else "local-variante-colab"

sys.path.insert(0, os.path.join(REPO, "nlp_agent"))
BASE_URL = os.environ.get("LLM_BASE_URL", "http://localhost:11434")

import httpx
for _ in range(60):                     # Ollama tarda unos segundos en abrir el puerto
    try:
        version_ollama = httpx.get(BASE_URL + "/api/version", timeout=3).json()["version"]
        break
    except httpx.HTTPError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama no respondió en 60 s" + (": ver /content/ollama.log" if EN_COLAB else ": bash scripts/ollama_wsl.sh iniciar"))
if version_ollama != OLLAMA_VERSION:
    raise RuntimeError(f"Ollama {version_ollama}: se necesita {OLLAMA_VERSION} para comparar con el equipo de desarrollo")

COMMIT = sh(["git", "-C", REPO, "rev-parse", "--short", "HEAD"])
fecha = datetime.now().strftime("%Y-%m-%d")
SALIDA = (Path("/content") / f"resultados_spike_{ENTORNO}_{datetime.now().strftime('%Y%m%d_%H%M')}" if EN_COLAB
          else Path(REPO) / "nlp_agent/evaluacion/resultados" / f"{fecha}_{ENTORNO}")
SALIDA.mkdir(parents=True, exist_ok=True)

print(f"Ollama {version_ollama} en {BASE_URL} · rama {RAMA} @ {COMMIT}")
print(f"Variante del conjunto: {VARIANTE} · resultados en {SALIDA}")
if EN_COLAB:
    time.sleep(3)
    gpu_ollama = [l for l in open("/content/ollama.log") if "inference compute" in l]
    print("GPU que ve Ollama:", gpu_ollama[-1].split("description=")[-1].split(" libdirs")[0] if gpu_ollama else "ninguna (correría en CPU)")

## 2. Modelos

En Colab se descargan los que falten; en el equipo no se descarga nada, porque el disco es limitado y ya están los tres del spike. Después se compara el **digest** de cada modelo con el del equipo de desarrollo. Mismo digest significa mismos pesos, que es la condición para comparar los resultados.

In [ ]:
MODELOS_EQUIPO = ["qwen3:8b", "qwen3:4b-instruct-2507-q4_K_M", "llama3.1:8b"]
MODELOS_TECHO = ["qwen3:14b", "qwen3:30b-a3b-instruct-2507-q4_K_M"]    # techo de calidad: solo con 24 GB de VRAM (L4)
INCLUIR_TECHO = EN_COLAB
QWEN_CON_RAZONAMIENTO = {"qwen3:8b", "qwen3:14b"}                     # W4: los demás no razonan
# Digests del equipo de desarrollo (16-oct-2026, Ollama 0.40.5)
DIGESTS_EQUIPO = {"qwen3:8b": "500a1f067a9f", "qwen3:4b-instruct-2507-q4_K_M": "0edcdef34593", "llama3.1:8b": "46e0c10c039e"}

MODELOS = MODELOS_EQUIPO + (MODELOS_TECHO if INCLUIR_TECHO else [])

def instalados():
    return {m["name"]: m for m in httpx.get(BASE_URL + "/api/tags", timeout=10).json()["models"]}

for m in MODELOS:
    if m in instalados() or not EN_COLAB:
        continue
    print(f"Descargando {m}…")
    with httpx.stream("POST", BASE_URL + "/api/pull", json={"model": m}, timeout=None) as r:
        ultimo = ""
        for linea in r.iter_lines():
            if not linea:
                continue
            d = json.loads(linea)
            if "error" in d:            # no seguir con otro modelo en silencio: se rompería la comparación
                raise RuntimeError(f"Ollama no pudo bajar {m}: {d['error']}")
            if d.get("status") and d["status"] != ultimo and not d["status"].startswith("pulling "):
                ultimo = d["status"]
                print(f"  {ultimo}")

inst = instalados()
DISPONIBLES = [m for m in MODELOS if m in inst]
for m in MODELOS:
    if m not in inst:
        print(f"{m:38} no está")
        continue
    digest = inst[m]["digest"][:12]
    comparable = "" if m not in DIGESTS_EQUIPO else ("· mismo digest que el equipo" if digest == DIGESTS_EQUIPO[m]
                                                      else f"· ⚠️ digest distinto al del equipo ({DIGESTS_EQUIPO[m]}): no comparable")
    print(f"{m:38} digest {digest}, {inst[m]['size'] / 1e9:.1f} GB {comparable}")

## 3. Explorar preguntas

Elige una o varias preguntas del conjunto (con Ctrl o Mayús) o escribe la tuya, y uno o varios modelos, y pulsa **Preguntar**. Para cada combinación verás:
- las herramientas que llamó el modelo, ronda por ronda;
- la respuesta final;
- la calificación (solo en las preguntas del conjunto);
- el tiempo y los tokens por ronda.

Las herramientas responden con los resultados sintéticos de cada pregunta. En una pregunta escrita por ti devuelven listas vacías: sirve para ver qué herramienta elige el modelo, no la respuesta.

In [ ]:
import html
import asyncio
import ipywidgets as w
from IPython.display import display, HTML

from core.config import Settings
from evaluacion.arnes import correr_pregunta
from evaluacion.conjunto import Pregunta, cargar, herramientas
from llm.cliente import ClienteOllama

conjunto = cargar(VARIANTE)
catalogo = herramientas(VARIANTE)
sucursal_admin = "NORTE" if VARIANTE == "colab" else "LA 21"

sel_preguntas = w.SelectMultiple(options=[(f"{p.id} · {p.pregunta}", p.id) for p in conjunto.preguntas],
                                 value=(conjunto.preguntas[0].id,), rows=10, description="Preguntas", layout=w.Layout(width="98%"))
libre = w.Text(placeholder="…o escribe otra pregunta (reemplaza la selección)", description="Libre", layout=w.Layout(width="98%"))
rol = w.Dropdown(options=[("Gerente", "gerente"), ("Adm. Bodega Central", "admin_bodega"),
                          (f"Adm. sucursal {sucursal_admin}", "admin_sucursal")], description="Rol (libre)")
sel_modelos = w.SelectMultiple(options=DISPONIBLES, value=tuple(DISPONIBLES[:1]), rows=len(DISPONIBLES), description="Modelos")
modo = w.ToggleButtons(options=[("Con GPU", "gpu"), ("Sin GPU", "cpu")], description="Modo")
razonamiento = w.Checkbox(description="Razonamiento (Qwen3 8B y 14B)")
boton = w.Button(description="Preguntar", button_style="primary", icon="play")
salida = w.Output()

def mostrar(contenido):
    # append_display_data y no `with salida:`: con código asíncrono, ipywidgets pierde la
    # captura después de un await y las tablas se van a otra celda o se pierden
    salida.append_display_data(HTML(contenido))

def _celda(c, pregunta_del_conjunto):
    llamadas = "<br>".join(f"{x['ronda']}. <code>{x['herramienta']}</code> {html.escape(json.dumps(x['argumentos'], ensure_ascii=False))}"
                           for x in c.llamadas) or "<i>ninguna</i>"
    respuesta = html.escape(c.texto).replace(chr(10), "<br>") or f"<b>error:</b> {c.error}"
    cal = ""
    if pregunta_del_conjunto:
        k = c.calificacion
        marcas = " ".join(f"{'✅' if k[x] else '❌'} {x}" for x in ("herramienta", "fidelidad", "hechos", "prohibido", "aclaracion", "rechazo", "sin_error"))
        cal = f"<p><b>{'ACIERTO' if k['acierto'] else 'FALLA'}</b> · {marcas}" + (
            f"<br>cifras sin respaldo: {k['cifras_sin_respaldo']}" if k["cifras_sin_respaldo"] else "") + "</p>"
    return (f"<td style='vertical-align:top;padding:6px'><p><b>Herramientas</b><br>{llamadas}</p>"
            f"<p><b>Respuesta</b><br>{respuesta}</p>{cal}"
            f"<p style='color:gray'>{c.ms_total / 1000:.1f} s · {c.rondas} rondas · {c.tokens_prompt_max} tokens por ronda</p></td>")

async def preguntar():
    salida.outputs = ()          # clear_output() se aplica tarde y borraría lo primero que se muestra
    try:
        if libre.value.strip():
            preguntas = [Pregunta(id="libre", rol=rol.value, sucursal_usuario={"gerente": None, "admin_bodega": "BODEGA_CENTRAL",
                         "admin_sucursal": sucursal_admin}[rol.value], tipo="abstencion", pregunta=libre.value.strip(), abstencion=True)]
            del_conjunto = False
        else:
            preguntas = [conjunto.por_id(i) for i in sel_preguntas.value]
            del_conjunto = True
        for p in preguntas:
            mostrar(f"<h4>{p.id} · {html.escape(p.pregunta)} <small>({p.rol}{', ' + p.sucursal_usuario if p.sucursal_usuario else ''})</small></h4>")
            if del_conjunto:
                esperado = " o ".join(f"<code>{e.herramienta}</code> {html.escape(json.dumps(e.argumentos, ensure_ascii=False))}" for e in p.esperado) or "ninguna herramienta"
                mostrar(f"<p style='color:gray'>Se espera: {esperado}" + (f" · nota: {html.escape(p.nota)}" if p.nota else "") + "</p>")
            columnas, encabezados = [], []
            for m in sel_modelos.value:
                cliente = ClienteOllama(Settings(OLLAMA_BASE_URL=BASE_URL, LLM_MODELO=m, LLM_TIMEOUT_S=300,
                                                 LLM_RAZONAMIENTO=razonamiento.value and m in QWEN_CON_RAZONAMIENTO))
                c = await correr_pregunta(cliente, p, catalogo, conjunto.fecha_datos,
                                          opciones={"num_gpu": 0} if modo.value == "cpu" else None)
                await cliente.cerrar()
                encabezados.append(f"<th style='text-align:left'>{m}</th>")
                columnas.append(_celda(c, del_conjunto))
            mostrar(f"<table style='width:100%;border-collapse:collapse'><tr>{''.join(encabezados)}</tr><tr>{''.join(columnas)}</tr></table>")
    except Exception as exc:     # sin esto, un error dentro del botón no se ve en ninguna parte
        mostrar(f"<p style='color:#b00'><b>Error:</b> {html.escape(repr(exc))}</p>")

boton.on_click(lambda _: asyncio.ensure_future(preguntar()))
display(sel_preguntas, libre, rol, sel_modelos, modo, razonamiento, boton, salida)

## 4. Examen completo, con resultados en CSV

Corre las preguntas en cada modelo y modo, y escribe en la carpeta de resultados:
- **`detalle.csv`**: una fila por corrida, con la pregunta, lo esperado, las herramientas, la respuesta y cada criterio (1 o 0);
- **`resumen.csv`**: una fila por modelo y modo, con el acierto, pass^3, la herramienta, la fidelidad, la abstención, las fugas, p50 y p95, y los tokens por ronda;
- **un `.jsonl` por modelo**, en el formato del repositorio, con la versión de Ollama, el digest del modelo y el commit.

Los CSV usan el formato del de INV-27: UTF-8 con BOM, «;» y coma decimal, y Excel en español los abre directo. Con GPU son unos minutos por modelo. **Sin GPU** (`"cpu"`) puede tardar horas: conviene `CORRIDAS = 1`, porque con temperatura 0 las tres corridas dan lo mismo.

In [ ]:
import csv
from datetime import timezone

from evaluacion import reporte

# ---------- Configuración ----------
MODELOS_EXAMEN = list(DISPONIBLES)  # o una lista, p. ej. ["qwen3:8b"]
PREGUNTAS = None                    # None = las 43; o una lista: ["v0-01", "v0-19"]
MODOS = ["gpu"]                     # agrega "cpu" para medir sin GPU (criterio 9; lento)
CORRIDAS = 3
RAZONAMIENTO = False                # True solo se aplica a qwen3:8b y qwen3:14b (W4)
TIMEOUT_S = 300
# -----------------------------------

preguntas = [p for p in conjunto.preguntas if PREGUNTAS is None or p.id in PREGUNTAS]
CRITERIOS = ["herramienta", "fidelidad", "hechos", "prohibido", "aclaracion", "rechazo", "sin_error", "acierto"]

def coma(x):
    """Número con coma decimal, como el CSV de INV-27."""
    return "" if x is None else (str(round(x, 3)).replace(".", ",") if isinstance(x, float) else str(x))

columnas = (["modelo", "digest", "modo", "razonamiento", "corrida", "id", "variante_de", "rol", "sucursal_usuario", "tipo", "pregunta",
             "esperado", "herramientas_usadas", "llamadas", "rondas", "respuesta", "error"]
            + CRITERIOS + ["cifras_sin_respaldo", "ms_total", "ms_modelo", "tokens_prompt_max", "tokens_respuesta", "nota"])
ruta_detalle = SALIDA / "detalle.csv"
nuevo = not ruta_detalle.exists()
archivos_jsonl = []

with open(ruta_detalle, "a", encoding="utf-8-sig" if nuevo else "utf-8", newline="") as f_csv:
    escritor = csv.writer(f_csv, delimiter=";")
    if nuevo:
        escritor.writerow(columnas)
    for modelo in MODELOS_EXAMEN:
        digest = instalados()[modelo]["digest"][:12]
        for modo_ex in MODOS:
            razona = RAZONAMIENTO and modelo in QWEN_CON_RAZONAMIENTO
            opciones = {"num_gpu": 0} if modo_ex == "cpu" else None
            cliente = ClienteOllama(Settings(OLLAMA_BASE_URL=BASE_URL, LLM_MODELO=modelo, LLM_TIMEOUT_S=TIMEOUT_S,
                                             LLM_RAZONAMIENTO=razona))
            # Calentamiento: carga el modelo antes de medir (su tiempo no cuenta en las preguntas)
            inicio = time.monotonic()
            await cliente.chat([{"role": "user", "content": "Responde solo: listo"}], opciones=opciones)
            carga_s = round(time.monotonic() - inicio, 1)
            print(f"=== {modelo} ({digest}) · {modo_ex}{' · razonamiento' if razona else ''} · carga {carga_s} s")
            ruta_jsonl = SALIDA / f"{modelo.replace(':', '_')}_{modo_ex}{'_razonamiento' if razona else ''}.jsonl"
            archivos_jsonl.append(ruta_jsonl)
            with open(ruta_jsonl, "w", encoding="utf-8") as f_jsonl:
                f_jsonl.write(json.dumps({
                    "meta": True, "modelo": modelo, "modelo_info": {"digest": digest}, "modo": modo_ex, "razonamiento": razona,
                    "corridas": CORRIDAS, "entorno": ENTORNO, "carga_s": carga_s, "ollama": version_ollama, "commit": COMMIT,
                    "conjunto": f"{conjunto.version} ({VARIANTE}, {len(preguntas)} preguntas)",
                    "inicio": datetime.now(timezone.utc).isoformat(timespec="seconds")}, ensure_ascii=False) + "\n")
                for n in range(1, CORRIDAS + 1):
                    for p in preguntas:
                        c = await correr_pregunta(cliente, p, catalogo, conjunto.fecha_datos, corrida=n, opciones=opciones)
                        f_jsonl.write(json.dumps(c.como_dict(), ensure_ascii=False) + "\n")
                        k = c.calificacion
                        escritor.writerow(
                            [modelo, digest, modo_ex, int(razona), n, p.id, p.variante_de or "", p.rol, p.sucursal_usuario or "",
                             p.tipo, p.pregunta,
                             " | ".join(f"{e.herramienta} {json.dumps(e.argumentos, ensure_ascii=False)}" for e in p.esperado) or "ninguna",
                             " > ".join(x["herramienta"] for x in c.llamadas) or "ninguna",
                             json.dumps(c.llamadas, ensure_ascii=False), c.rondas, c.texto, c.error or ""]
                            + [int(bool(k[x])) for x in CRITERIOS]
                            + [", ".join(k["cifras_sin_respaldo"]), coma(c.ms_total), coma(c.ms_modelo),
                               c.tokens_prompt_max, c.tokens_respuesta, p.nota or ""])
                        f_csv.flush()
                        f_jsonl.flush()
                        print(f"  [{n}/{CORRIDAS}] {p.id:7} {'OK ' if k['acierto'] else '-- '}"
                              f"herr={'s' if k['herramienta'] else 'n'} fid={'s' if k['fidelidad'] else 'n'} "
                              f"{c.ms_total / 1000:6.1f} s {c.error or ''}")
            await cliente.cerrar()

# Resumen de todo lo que hay en la carpeta (también de corridas anteriores en esta sesión)
datos = reporte.leer(sorted(SALIDA.glob("*.jsonl")))
campos = ["preguntas", "corridas", "acierto", "pass3", "herramienta", "pass3_herramienta", "fidelidad", "abstencion",
          "fugas", "errores", "razonamiento_filtrado", "p50_s", "p95_s", "tokens_ronda_p95", "tokens_ronda_max"]
with open(SALIDA / "resumen.csv", "w", encoding="utf-8-sig", newline="") as f:
    escritor = csv.writer(f, delimiter=";")
    escritor.writerow(["modelo", "digest", "modo", "razonamiento", "ollama", "commit", "carga_s"] + campos)
    for d in datos:
        r, m = reporte.resumen(d["corridas"]), d["meta"]
        escritor.writerow([m["modelo"], m.get("modelo_info", {}).get("digest", ""), m["modo"], int(m.get("razonamiento", False)),
                           m.get("ollama", ""), m.get("commit", ""), coma(m.get("carga_s"))] + [coma(r[x]) for x in campos])

from IPython.display import Markdown
print(f"Detalle: {ruta_detalle}\nResumen: {SALIDA / 'resumen.csv'}")
display(Markdown(reporte.markdown(datos)))

## 5. Rendimiento (criterio 3)

Para cada modelo mide:
- la carga en frío;
- la latencia al primer token;
- la velocidad de lectura del prompt, sin contar lo que Ollama reutiliza;
- la generación;
- la memoria.

Se mide sobre la respuesta final que ve el usuario. La última línea mide la latencia de Qwen3 con razonamiento (W4).

In [ ]:
import argparse
from evaluacion import smoke

for m in MODELOS_EXAMEN:
    for modo_smoke in MODOS:
        smoke.correr(argparse.Namespace(modelo=m, modo=modo_smoke, razonamiento=False, variante=VARIANTE,
                                        entorno=ENTORNO, base_url=BASE_URL, salida=str(SALIDA)))
if "qwen3:8b" in MODELOS_EXAMEN:
    smoke.correr(argparse.Namespace(modelo="qwen3:8b", modo="gpu", razonamiento=True, variante=VARIANTE,
                                    entorno=ENTORNO, base_url=BASE_URL, salida=str(SALIDA)))

## 6. Descargar los resultados

En Colab, descarga un zip con la carpeta de resultados (los CSV, los `.jsonl` y el rendimiento) para versionarla en el repositorio (N9). En el equipo ya quedan en `nlp_agent/evaluacion/resultados/`.

In [ ]:
if EN_COLAB:
    from google.colab import files
    shutil.make_archive(str(SALIDA), "zip", SALIDA)
    files.download(str(SALIDA) + ".zip")
else:
    print("Resultados en", SALIDA)